# 📊 Notebook 03: NASA GRACE/GRACE-FO Data Extraction

**Objective:** Download and process Terrestrial Water Storage (TWS) anomalies from NASA GRACE satellite data.

**Source:** [NASA GRACE Tellus](https://grace.jpl.nasa.gov/) via [PO.DAAC](https://podaac.jpl.nasa.gov/)

**Collection:** `TELLUS_GRAC-GRFO_MASCON_CRI_GRID_RL06.3_V4`

**Key Variables:**
- Terrestrial Water Storage (TWS) Anomaly (cm)
- Groundwater Storage Anomaly
- Trend rates (cm/year)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import logging
import warnings
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

import config
from src.extractors.grace_extractor import GraceExtractor

plt.style.use('dark_background')
print('✅ Setup complete')

## 1. Data Download

Uses `podaac-data-downloader` for real GRACE data. If not installed or not authenticated,
synthetic data mimicking real GRACE patterns will be generated.

**To download real data:**
```bash
pip install podaac-data-subscriber
podaac-data-downloader -c TELLUS_GRAC-GRFO_MASCON_CRI_GRID_RL06.3_V4 \
    -d ./data/raw/grace \
    --start-date 2002-04-04T00:00:00Z \
    --end-date 2024-12-31T00:00:00Z \
    -e .nc
```

In [ ]:
extractor = GraceExtractor()

# Attempt download (falls back to synthetic)
extractor.download_data()
print('\n✅ Data ready')

## 2. Data Loading & Exploration

In [ ]:
# Extract features with rolling statistics
df = extractor.extract_features()

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nCountries: {df["country_code"].nunique()}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
df.head(10)

In [ ]:
# Summary statistics
df.describe().round(3)

## 3. TWS Anomaly Visualization

In [ ]:
# TWS anomaly time series for most depleted regions
depleted = extractor.get_most_depleted_regions(df, n=8)
print('Most depleted regions:')
print(depleted.to_string(index=False))

top_codes = depleted['country_code'].tolist()
plot_data = df[df['country_code'].isin(top_codes)].copy()
plot_data['date'] = pd.to_datetime(plot_data['date'])

fig = px.line(
    plot_data, x='date', y='tws_anomaly_cm',
    color='country', title='TWS Anomaly — Most Depleted Regions',
    labels={'tws_anomaly_cm': 'TWS Anomaly (cm)', 'date': 'Date'},
)
fig.update_layout(template='plotly_dark', height=500)
fig.show()

In [ ]:
# Multi-panel: TWS anomaly for selected countries
countries_to_plot = ['IND', 'SAU', 'IRN', 'BRA', 'USA', 'AUS']

fig = make_subplots(
    rows=3, cols=2, shared_xaxes=True,
    subplot_titles=[df[df['country_code']==c]['country'].iloc[0] if c in df['country_code'].values else c 
                    for c in countries_to_plot],
    vertical_spacing=0.08,
)

for i, code in enumerate(countries_to_plot):
    row, col = i // 2 + 1, i % 2 + 1
    region = df[df['country_code'] == code].copy()
    region['date'] = pd.to_datetime(region['date'])
    region = region.sort_values('date')
    
    # TWS anomaly
    fig.add_trace(
        go.Scatter(x=region['date'], y=region['tws_anomaly_cm'],
                   mode='lines', name=code, showlegend=False,
                   line=dict(width=1.5),
                   fill='tozeroy'),
        row=row, col=col
    )
    
    # 12-month moving average
    if 'tws_12month_avg' in region.columns:
        fig.add_trace(
            go.Scatter(x=region['date'], y=region['tws_12month_avg'],
                       mode='lines', name=f'{code} (12m avg)', showlegend=False,
                       line=dict(width=2, dash='dash', color='white')),
            row=row, col=col
        )

fig.update_layout(
    title='Terrestrial Water Storage Anomalies by Country',
    template='plotly_dark', height=700,
)
fig.update_yaxes(title_text='TWS (cm)', col=1)
fig.show()

In [ ]:
# Global TWS trends map
if 'tws_trend_cm_yr' in df.columns:
    trend_data = df.groupby(['country_code', 'country']).agg({
        'tws_trend_cm_yr': 'first',
        'tws_anomaly_cm': 'mean',
    }).reset_index()
    
    fig = px.choropleth(
        trend_data, locations='country_code', locationmode='ISO-3',
        color='tws_trend_cm_yr', hover_name='country',
        color_continuous_scale='RdBu',
        color_continuous_midpoint=0,
        title='TWS Trend (cm/year) — Blue=Gaining, Red=Losing Water',
        labels={'tws_trend_cm_yr': 'Trend (cm/yr)'},
    )
    fig.update_layout(
        geo=dict(showframe=False, projection_type='natural earth'),
        template='plotly_dark', height=500,
    )
    fig.show()

In [ ]:
# Groundwater anomaly vs TWS anomaly scatter
if 'groundwater_anomaly_cm' in df.columns:
    sample = df.sample(min(2000, len(df)), random_state=42)
    
    fig = px.scatter(
        sample, x='tws_anomaly_cm', y='groundwater_anomaly_cm',
        color='country', opacity=0.6,
        title='TWS vs Groundwater Anomaly',
        labels={'tws_anomaly_cm': 'TWS Anomaly (cm)', 'groundwater_anomaly_cm': 'Groundwater Anomaly (cm)'},
    )
    fig.update_layout(template='plotly_dark', height=500)
    fig.show()

## 4. Save Processed Data

In [ ]:
output_path = extractor.save_processed(df)
print(f'✅ Saved to: {output_path}')
print(f'   Shape: {df.shape}')
print(f'   Countries: {df["country_code"].nunique()}')
print(f'   Date range: {df["date"].min()} to {df["date"].max()}')